# 02_1 — Xuất dữ liệu sau tiền xử lý (trước khi vào train)

Notebook này **không train**. Nó chụp lại dữ liệu ở đúng điểm `train_final()` nhận vào và ghi
ra **một file duy nhất** để xem.

Trong `03_train_model.ipynb`, `train_final()` chỉ làm hai việc trước khi train:

```python
encoder = fit_encoder(frame, features)                    # học vocab
x_all, feature_names, _ = apply_encoder(frame, encoder)   # 40 cột -> 516 cột số
models[head] = make_model().fit(x_all, frame[head])       # <- CHỤP NGAY TRƯỚC DÒNG NÀY
```

Có **hai dạng** của cùng một dữ liệu ở điểm này:

| Dạng | Cột | Đọc được? | Xuất |
|---|---:|---|---|
| **Thô** — `frame[features]`, đầu vào của `fit_encoder` | 40 | có: `Tuya`, `googlecast`, `0303\|...` | **mặc định** |
| Đã encode — `x_all`, mảng thật truyền vào `.fit()` | 516 | không: `0.0`, `0.317`, `-1.0` | bật `INCLUDE_ENCODED` |

Đầu vào: `Data/sessions_verified.parquet` (notebook 02).
Đầu ra: `Reports/train_input_<timestamp>_<feature_set>.csv`.

In [1]:
# Nạp định nghĩa từ 03 mà không chạy train — cùng cách notebook 02 vẫn làm.
from pathlib import Path

_cwd = Path.cwd().resolve()
_ROOT = next((p for p in (_cwd, *_cwd.parents)
              if (p / "Code" / "03_train_model.ipynb").is_file()), None)
assert _ROOT is not None, f"Không thấy Code/03_train_model.ipynb quanh {_cwd}"
_CODE = _ROOT / "Code"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_CODE / "03_train_model.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 80)

print("Python :", __import__("sys").executable)
print("Dataset:", SESSIONS_PATH)

Gốc dự án: D:\12.VQEC\02.SDC


Đã nạp hàm SDC từ 03_train_model.ipynb
Python : C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\python.exe
Dataset: D:\12.VQEC\02.SDC\Data\sessions_verified.parquet


## Tham số

`FEATURE_SET` chọn bộ feature muốn chụp:

- `all` — toàn bộ 40 cột, đúng thứ model sản phẩm đang dùng.
- `no_content` — bỏ `dns_tokens` và `tls_sni_tokens`, tức kịch bản DoH/DoT bật và ECH bật.

`INCLUDE_ENCODED` mặc định tắt: bật thì file có thêm 516 cột số sau TF-IDF/ordinal, nặng gấp
khoảng 4 lần và không đọc được bằng mắt.

In [2]:
FEATURE_SET = "all"        # "all" | "no_content"
INCLUDE_ENCODED = False    # True -> ghi kèm 516 cột đã encode
FORMAT = "csv"             # "csv" | "parquet"
OUT_FILE = None            # None -> Reports/train_input_<timestamp>_<FEATURE_SET>.<FORMAT>

## 1. Nạp dataset và kiểm tra như lúc train

`load_sessions` đã chạy sẵn `assert_serveable` — không feature nào vượt quá telemetry driver
cung cấp. `validate_training_frame` là cửa kiểm tra mà `train_main` đi qua, gọi lại ở đây để
dữ liệu chụp được đúng là dữ liệu đủ điều kiện train, không phải một bản khác.

In [3]:
frame, feature_groups = load_sessions(SESSIONS_PATH)
validate_training_frame(frame)

features = feature_groups[FEATURE_SET]
meta_cols = [c for c in META_COLS if c in frame.columns]

print(f"{len(frame)} session × {frame.shape[1]} cột")
print(f"{frame.canonical_device.nunique()} thiết bị, {frame.date.nunique()} ngày")
print(f"\nBộ feature '{FEATURE_SET}': {len(features)} cột thô")
for group in ("num", "cat", "text"):
    cols = [c for c in feature_groups[group] if c in features]
    print(f"  {group:5s} {len(cols):3d}  {cols if group != 'num' else ''}")

8139 session × 49 cột
39 thiết bị, 34 ngày

Bộ feature 'all': 40 cột thô
  num    31  
  cat     5  ['dhcp_prl', 'tls_fp', 'tls_version', 'tls_alpn', 'tls_ciphers']
  text    4  ['dhcp_vci', 'dns_tokens', 'mdns_tokens', 'tls_sni_tokens']


## 2. Bốn mươi cột feature thô

Đây là thứ `fit_encoder` nhận vào. Bảng dưới liệt kê từng cột: thuộc nhóm nào, từ nguồn nào,
bao nhiêu phần trăm session có giá trị thật, và một giá trị ví dụ.

`coverage` tính khác nhau theo nhóm, vì "rỗng" ở mỗi nhóm là một thứ khác nhau:
`text`/`cat` rỗng là `<missing>` (thiết bị không nói giao thức đó), còn `num` rỗng là `0`
(cờ option không bật). Missing là **trạng thái hợp lệ**, không impute — thiết bị không nói
TLS là một đặc điểm nhận dạng thật, nên cột `has_*` giữ lại đúng thông tin đó.

In [4]:
group_of = {c: g for g in ("num", "cat", "text") for c in feature_groups[g]}


def coverage(col):
    # Tỉ lệ session có giá trị thật. Text/cat: khác <missing>. Num: khác 0.
    series = frame[col]
    if group_of[col] == "num":
        return float((series != 0).mean())
    return float((series.astype(str) != MISSING).mean())


def sample_value(col):
    real = frame.loc[frame[col].astype(str) != MISSING, col]
    return str(real.iloc[0]) if len(real) else MISSING


raw_features = pd.DataFrame({
    "feature": features,
    "group": [group_of[c] for c in features],
    "source": [source_of(c) for c in features],
    "dtype": [str(frame[c].dtype) for c in features],
    "n_unique": [int(frame[c].nunique()) for c in features],
    "coverage": [coverage(c) for c in features],
    "example": [sample_value(c) for c in features],
}).sort_values(["group", "source", "feature"]).reset_index(drop=True)

display(raw_features)

,feature,group,source,dtype,n_unique,coverage,example
0,dhcp_prl,cat,dhcp,object,17,0.107384,"1,3,4,5,6,12,15,28,42"
1,tls_alpn,cat,tls,object,5,0.299054,http/1.1
2,tls_ciphers,cat,tls,object,30,0.707458,"49200,49196,49192,49188,49172,49162,165,163,161,159,107,106,105,104,57,56,55..."
3,tls_fp,cat,tls,object,36,0.707458,"0303|49200,49196,49192,49188,49172,49162,165,163,161,159,107,106,105,104,57,..."
4,tls_version,cat,tls,object,3,0.707458,0303
5,dhcp_opt_1,num,dhcp,int64,2,0.107384,0
6,dhcp_opt_119,num,dhcp,int64,2,0.002826,0
7,dhcp_opt_12,num,dhcp,int64,2,0.056518,0
8,dhcp_opt_121,num,dhcp,int64,2,0.031945,0
9,dhcp_opt_15,num,dhcp,int64,2,0.100258,0


## 3. Nội dung thật của các cột chuỗi

Bảng trên chỉ cho một giá trị ví dụ mỗi cột. Chín cột `cat` và `text` là nơi chứa gần hết
thông tin nhận dạng, nên xem thẳng các giá trị hay gặp nhất của chúng.

In [5]:
for col in [c for c in features if group_of[c] in ("cat", "text")]:
    counts = frame[col].astype(str).value_counts()
    real = counts[counts.index != MISSING]
    print(f"\n=== {col}  ({group_of[col]}, {counts.size} giá trị, "
          f"phủ {coverage(col):.1%}) ===")
    for value, count in real.head(5).items():
        shown = value if len(value) <= 100 else value[:97] + "..."
        print(f"  {count:5d}  {shown}")
    if len(real) > 5:
        print(f"  ... còn {len(real) - 5} giá trị")


=== dhcp_prl  (cat, 17 giá trị, phủ 10.7%) ===
    404  1,3,6,12,15,28,42
    252  1,3,28,6,15,44,46,47,31,33,121,43
     52  1,3,28,6
     32  1,33,3,6,15,28,51,58,59
     26  1,3,6,12,15,28,40,41,42
  ... còn 11 giá trị

=== dhcp_vci  (text, 15 giá trị, phủ 6.0%) ===
    338  udhcp 1.24.1
     26  udhcp 0.9.9-pre
     21  udhcp 1.30.1
     15  dhcpcd-5.5.6
     15  dhcpcd-6.8.2:Linux-4.4.22+:armv7l:MT8167B
  ... còn 9 giá trị

=== dns_tokens  (text, 1268 giá trị, phủ 88.8%) ===
   1852  a3 tuyaus com
    317  dh amcrestsecurity com
    308  device metrics us 2 amazon com
    301  apicom netatmo net
    298  mcs arlo com
  ... còn 1262 giá trị

=== mdns_tokens  (text, 209 giá trị, phủ 22.7%) ===
    328  _sonos _tcp local
    323  _dcp _tcp local
    226  _sengled _udp local
    199  _viziocast _tcp local _amzn alexa _matter _matterc _udp _ipp
    176  _sengled _udp local _viziocast _tcp _amzn alexa _matter _matterc _ipp
  ... còn 203 giá trị

=== tls_fp  (cat, 36 giá trị, phủ 70.7%)

## 4. Kiểm tra nhanh trước khi ghi

Encode thử để biết 40 cột thô nở ra bao nhiêu cột số, và để chạy phép kiểm tra đáng giá nhất
trên dữ liệu này: hai session có vector đầu vào **giống hệt nhau** nhưng nhãn khác nhau thì
không model nào dùng bộ feature này tách được chúng — đó là sàn lỗi cố định, không phải lỗi
của model.

In [6]:
encoder = fit_encoder(frame, features)
x, feature_names, _ = apply_encoder(frame, encoder)

n_num, n_cat = len(encoder["num_cols"]), len(encoder["cat_cols"])
print(f"{len(features)} cột thô -> ma trận {x.shape[0]} × {x.shape[1]} ({x.dtype})")
print(f"  num  {n_num:3d} -> {n_num:3d}  passthrough")
print(f"  cat  {n_cat:3d} -> {n_cat:3d}  OrdinalEncoder, OOV={OOV_INDEX}")
print(f"  text {len(encoder['text_cols']):3d} -> {len(feature_names) - n_num - n_cat:3d}  TF-IDF")
for col, vec in encoder_vectorizers(encoder).items():
    print(f"         {col:16s} {len(vec.vocabulary_):4d} / {TFIDF_MAX_FEATURES[col]:4d} token")

print("\nPhân bố nhãn:")
for head in LABEL_COLS:
    counts = frame[head].astype(str).value_counts()
    rare = sorted(rare_classes(frame[head].astype(str)))
    print(f"  {head:6s} {counts.size:2d} lớp  |  lớn nhất {counts.index[0]} ({counts.iloc[0]})"
          f"  |  hiếm (<{RARE_THRESHOLD}): {rare or 'không có'}")

print("\nĐộ phủ nguồn:")
for s in SOURCES:
    print(f"  has_{s:5s} {frame[f'has_{s}'].mean():6.1%}")

_, inverse = np.unique(x, axis=0, return_inverse=True)
tagged = frame.assign(_group=inverse)
print(f"\n{len(frame)} session -> {len(np.unique(inverse))} vector khác nhau")
for head in LABEL_COLS:
    per_group = tagged.groupby("_group")[head].nunique()
    stuck = int(tagged._group.isin(per_group[per_group > 1].index).sum())
    print(f"  {head:6s} {stuck:4d} session ({stuck / len(frame):.1%}) trùng vector nhưng lệch nhãn")

40 cột thô -> ma trận 8139 × 516 (float32)
  num   31 ->  31  passthrough
  cat    5 ->   5  OrdinalEncoder, OOV=-1
  text   4 -> 480  TF-IDF
         dhcp_vci           30 /  100 token
         dns_tokens        200 /  200 token
         mdns_tokens       100 /  100 token
         tls_sni_tokens    150 /  150 token

Phân bố nhãn:
  make   18 lớp  |  lớn nhất Tuya ODM (2121)  |  hiếm (<10): ['LG']
  type   12 lớp  |  lớn nhất IP Camera (2471)  |  hiếm (<10): ['Television', 'Weather Station']
  model  26 lớp  |  lớn nhất Tuya Plug (2121)  |  hiếm (<10): ['LG Smart TV', 'Netatmo Weather Station']

Độ phủ nguồn:
  has_dhcp   10.7%
  has_dns    88.8%
  has_mdns   22.7%
  has_tls    70.7%



8139 session -> 759 vector khác nhau
  make      0 session (0.0%) trùng vector nhưng lệch nhãn
  type      0 session (0.0%) trùng vector nhưng lệch nhãn
  model     3 session (0.0%) trùng vector nhưng lệch nhãn


## 5. Ghi file

Một file, không kèm file phụ: meta, 3 nhãn, rồi 40 cột feature thô. Index là `session_id` nên
nối ngược được về `Data/sessions_verified.parquet` khi cần.

In [7]:
train_input = frame[meta_cols + LABEL_COLS + list(features)]

if INCLUDE_ENCODED:
    encoded = pd.DataFrame(x, columns=[f"enc::{n}" for n in feature_names], index=frame.index)
    train_input = pd.concat([train_input, encoded], axis=1)

suffix = "csv" if FORMAT == "csv" else "parquet"
out_file = Path(OUT_FILE) if OUT_FILE else (
    REPORTS / f"train_input_{datetime.now():%Y%m%d_%H%M%S}_{FEATURE_SET}.{suffix}")
out_file.parent.mkdir(parents=True, exist_ok=True)

if FORMAT == "csv":
    train_input.to_csv(out_file, encoding="utf-8-sig")   # BOM để Excel đọc đúng tiếng Việt
else:
    train_input.to_parquet(out_file)

print(f"Đã ghi {out_file}")
print(f"  {train_input.shape[0]} dòng × {train_input.shape[1]} cột "
      f"({len(meta_cols)} meta + {len(LABEL_COLS)} nhãn + {len(features)} feature thô"
      f"{f' + {len(feature_names)} encoded' if INCLUDE_ENCODED else ''}), "
      f"{out_file.stat().st_size / 1024**2:.1f} MB")
display(train_input.head(5))

Đã ghi D:\12.VQEC\02.SDC\Reports\train_input_20260914_143123_all.csv
  8139 dòng × 49 cột (6 meta + 3 nhãn + 40 feature thô), 4.2 MB


,canonical_device,scenario,capture_file,date,mac,n_sources,make,type,model,dhcp_prl,...,mdns_tokens,tls_fp,tls_version,tls_alpn,tls_ciphers,tls_sni_tokens,has_dhcp,has_dns,has_mdns,has_tls
session_id,,,,,,,,,,,,,,,,,,,,,
AMCREST WiFi Camera_2021-11-02_00,Amcrest WiFi Camera,IDLE,2021_11_02_Idle.pcap,2021-11-02,9c:8e:cd:1d:ab:9f,1,Amcrest,IP Camera,Amcrest WiFi Camera,<missing>,...,<missing>,<missing>,<missing>,<missing>,<missing>,<missing>,0,1,0,0
AMCREST WiFi Camera_2021-11-02_01,Amcrest WiFi Camera,IDLE,2021_11_02_Idle.pcap,2021-11-02,9c:8e:cd:1d:ab:9f,1,Amcrest,IP Camera,Amcrest WiFi Camera,<missing>,...,<missing>,<missing>,<missing>,<missing>,<missing>,<missing>,0,1,0,0
AMCREST WiFi Camera_2021-11-02_02,Amcrest WiFi Camera,IDLE,2021_11_02_Idle.pcap,2021-11-02,9c:8e:cd:1d:ab:9f,1,Amcrest,IP Camera,Amcrest WiFi Camera,<missing>,...,<missing>,<missing>,<missing>,<missing>,<missing>,<missing>,0,1,0,0
AMCREST WiFi Camera_2021-11-02_03,Amcrest WiFi Camera,IDLE,2021_11_02_Idle.pcap,2021-11-02,9c:8e:cd:1d:ab:9f,1,Amcrest,IP Camera,Amcrest WiFi Camera,<missing>,...,<missing>,<missing>,<missing>,<missing>,<missing>,<missing>,0,1,0,0
AMCREST WiFi Camera_2021-11-02_04,Amcrest WiFi Camera,IDLE,2021_11_02_Idle.pcap,2021-11-02,9c:8e:cd:1d:ab:9f,1,Amcrest,IP Camera,Amcrest WiFi Camera,<missing>,...,<missing>,<missing>,<missing>,<missing>,<missing>,<missing>,0,1,0,0
